# Psychometrics: Inferring Latent Constructs via Factor Analysis, IRT, and Structural Equation Modeling

In social science, the things we truly care about—anxiety, ability, political attitudes, organizational commitment—are almost impossible to observe directly. Psychometrics addresses precisely this dilemma: latent constructs are invisible, and all we have are the "noisy indicators" they leave behind—questionnaire items, test responses, scale items. This notebook walks through the complete machinery of "inferring invisible constructs from observable items", unfolding three complementary perspectives layer by layer.

The first perspective is **Confirmatory Factor Analysis (CFA)**. When you already have a theory and have predetermined "which items measure the same thing", CFA lets you formalize this assumption as a measurement model, estimate each item's **factor loading**, and judge model-data fit using the three-metric suite `CFI / RMSEA / SRMR`. It differs from exploratory EFA—EFA lets the data tell you how many factors exist, while CFA lets you posit a structure first, then test it against data. The second perspective is **Item Response Theory (IRT)**. CFA treats responses as continuous variables, but responses are actually binary—either right or wrong. IRT acknowledges this, using three sets of parameters—each item's **discrimination a**, **difficulty b**, and each person's **ability θ**—to model the probability of answering an item correctly as a logistic curve, and to calculate how much information each item provides at different ability levels. The third perspective is **Structural Equation Modeling (SEM)**. While the first two ask "do these items measure the same construct", SEM goes further, asking about **paths between constructs**—it treats a set of regression equations as a simultaneous system to estimate, yielding standardized path coefficients, R² for each equation, and global fit indices.

Each method has its own assumptions and pitfalls: CFA depends on correctly pre-specifying the factor structure (if wrong, fit indices will tell you); IRT's maximum likelihood can push the discrimination parameter a to a boundary when item discrimination is very high, which requires attention; SEM's path analysis here involves only observed variables, not latent variable estimation. To make each step **falsifiable**, all three datasets we use are synthetic data with known parameters: IRT data contains the true a and b for each item, multilevel data contains the true slope β=2.0. This way each step can be validated by comparing estimates to ground truth, verifying not "does it look like a good fit" but "does it recover what is known to be true".

We execute the complete workflow using `socialverse`, which benchmarks against R's **lavaan** (CFA/SEM), **mirt** (IRT), **psych** and Python's **semopy** / **factor_analyzer** / **girth**—it prioritizes calling these specialized packages when available, falling back to genuine implementations based on statsmodels / scipy / numpy when they are not, so the numbers below can be computed even without any psychometrics-specific packages installed.

In [1]:
import os
import sys

# 确保 import 到本 worktree 的 socialverse(而非环境里另一份旧安装):把 worktree 根插到 sys.path 最前。
_here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_root = os.path.abspath(os.path.join(_here, ".."))
if _root not in sys.path:
    sys.path.insert(0, _root)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # 无显示后端:图直接写文件
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import spearmanr, pearsonr

# 让图里的中文正常渲染:挑一个系统里存在的 CJK 字体。
_installed = {f.name for f in fm.fontManager.ttflist}
for _cjk in ("PingFang SC", "PingFang HK", "Hiragino Sans GB", "Songti SC",
             "STHeiti", "Arial Unicode MS", "Noto Sans CJK SC", "WenQuanYi Zen Hei"):
    if _cjk in _installed:
        plt.rcParams["font.sans-serif"] = [_cjk, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False  # 负号用 ASCII,避免缺字

import socialverse as sv
from socialverse import datasets as ds


# 图存到 notebook 同目录(用相对文件名引用),不论从哪个 cwd 运行都对得上。
def figpath(name: str) -> str:
    return os.path.join(_here, name)

## Loading Data

We use two built-in synthetic datasets. The first, `load_irt()`, returns a pair of results: a 400 person × 10 item binary response matrix `resp`, and a truth table `truth` recording the true discrimination a and difficulty b used to generate each item. This response data serves double duty—fed as continuous indicators to CFA and as binary responses to IRT. The second, `load_multilevel()`, is two-level data with students nested in schools, with columns `school / student / x / y`, generated from `y = 1 + u + 2·x + ε`, true slope β=2.0, which we use for SEM's path analysis.

First load both datasets and take a look. Note that the difficulties b in the `truth` table are intentionally spread uniformly from −2 to +2, so when we later recover the difficulty ordering with IRT, success or failure is immediately obvious.

In [2]:
resp, truth = ds.load_irt()
multilevel = ds.load_multilevel()

print("IRT 作答矩阵:", resp.shape, "| 列:", list(resp.columns))

IRT 作答矩阵: (400, 10) | 列: ['item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'item7', 'item8', 'item9', 'item10']


True parameters embedded in each item (a = discrimination, b = difficulty, sorted by difficulty in ascending order):

In [3]:
truth

,item,a,b
0,item1,0.925,-2.000
1,item2,1.462,-1.556
2,item3,1.946,-1.111
3,item4,1.631,-0.667
4,item5,1.975,-0.222
5,item6,1.447,0.222
6,item7,1.978,0.667
7,item8,1.788,1.111
8,item9,1.712,1.556
9,item10,1.855,2.000


First few rows of multilevel data, with true structure `y = 1 + u + 2·x + ε` (`u` is the school random effect):

In [4]:
multilevel.head()

,school,student,x,y
0,0,0,-0.1321,1.5019
1,0,1,0.1049,0.7999
2,0,2,0.3616,3.1529
3,0,3,0.9471,2.3162
4,0,4,-1.2654,-2.0284


## CFA: Does the Two-Factor Measurement Model Fit the Data

First question: can these 10 items be cleanly grouped into two latent factors? We posit a specific structure—items 1–5 load on factor F1, items 6–10 load on factor F2—and let CFA test this hypothesized structure. Because it is "confirmatory", factor assignments are fixed by us, not derived from data; data's role is to answer "does your structure fit".

We feed the 0/1 response matrix as continuous indicators, using `model_spec` to specify which items load on each factor. When `semopy` is available, `sv.tl.cfa` runs lavaan-style ML-SEM; when not, it performs single-factor ML analysis on each factor block, then constructs the model-implied covariance matrix for scoring—this is honestly flagged in the returned `note`.

In [5]:
st_cfa = sv.StudyState()
st_cfa.write("sources", "datasets", resp.astype(float))  # CFA 把作答当连续指标

cols = list(resp.columns)
spec = {"F1": cols[:5], "F2": cols[5:]}   # 事先写死的测量模型:前5题→F1,后5题→F2
sv.tl.cfa(st_cfa, model_spec=spec)

cfa_model = st_cfa.models["cfa"]
fit = st_cfa.diagnostics["fit_indices"]
print("backend:", cfa_model["backend"])
print("note   :", cfa_model["note"])

backend: path_ml_statsmodels
note   : block-wise ML loadings + estimated Φ (honest approximation)


First, look at **factor loadings**. A loading measures "how strongly an item correlates with its target factor"; standardized loadings above 0.4 are typically considered acceptable. Arrange the loadings of both factors in a table; all items show positive loadings and most exceed 0.4, indicating items reliably converge on their respective factors.

In [6]:
loading_rows = []
for factor, loads in cfa_model["loadings"].items():
    for item, v in loads.items():
        loading_rows.append({"因子": factor, "题目": item, "载荷": round(v, 3)})
pd.DataFrame(loading_rows)

,因子,题目,载荷
0,F1,item1,0.403
1,F1,item2,0.499
2,F1,item3,0.506
3,F1,item4,0.551
4,F1,item5,0.566
5,F2,item6,0.605
6,F2,item7,0.540
7,F2,item8,0.556
8,F2,item9,0.432
9,F2,item10,0.304


Next, examine the correlation between the two factors and the three global fit indices. The interfactor correlation Φ tells us whether F1 and F2 are independent constructs (here approximately 0.5, moderate correlation, reasonable); `CFI / RMSEA / SRMR` comprise the standard three-metric suite for judging overall model quality, each with established thresholds.

In [7]:
fc = np.array(fit["factor_correlation"])
print(f"因子间相关 Φ(F1, F2) = {fc[0, 1]:+.3f}")
print(f"平均载荷 = {cfa_model['mean_loading']:.3f} | 正载荷比例 = {cfa_model['prop_positive']:.0%}")
print()
print(f"CFI   = {fit['CFI']:.3f}   (≥ 0.95 为良好)")
print(f"RMSEA = {fit['RMSEA']:.3f}   (≤ 0.06 为良好)")
print(f"SRMR  = {fit['SRMR']:.3f}   (≤ 0.08 为良好)")
print(f"χ²({fit['df']:.0f}) = {fit['chi2']:.2f}, n = {fit['n']}")

因子间相关 Φ(F1, F2) = +0.506
平均载荷 = 0.496 | 正载荷比例 = 100%

CFI   = 0.950   (≥ 0.95 为良好)
RMSEA = 0.042   (≤ 0.06 为良好)
SRMR  = 0.059   (≤ 0.08 为良好)
χ²(34) = 58.12, n = 400


The three indices—CFI≈0.95, RMSEA≈0.04, SRMR≈0.06—all fall within acceptable ranges, indicating that the hypothesized two-factor structure is consistent with the data and our assertion is supported. Below we plot the loadings of both factors as bar charts to visualize each item's loading strength on its target factor; the dashed line marks the 0.4 reference threshold.

In [8]:
fig, ax = plt.subplots(figsize=(8, 4.5))
factors = list(cfa_model["loadings"].keys())
palette = {"F1": "#3B6FB6", "F2": "#C1544A"}
pos = 0
for factor in factors:
    loads = cfa_model["loadings"][factor]
    xs = np.arange(pos, pos + len(loads))
    ax.bar(xs, list(loads.values()), color=palette[factor], label=factor, width=0.75)
    for x, it in zip(xs, loads.keys()):
        ax.text(x, -0.03, it, ha="center", va="top", fontsize=8, rotation=45)
    pos += len(loads) + 1  # 两个因子之间留一格空隙

ax.axhline(0, color="black", lw=0.8)
ax.axhline(0.4, color="gray", ls="--", lw=0.8, label="载荷 0.4 参考线")
ax.set_ylabel("standardized loading")
ax.set_title(f"CFA 两因子测量模型载荷  (CFI={fit['CFI']:.2f}, "
             f"RMSEA={fit['RMSEA']:.2f}, SRMR={fit['SRMR']:.2f})")
ax.set_xticks([])
ax.set_ylim(-0.15, 1.0)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(figpath("fig_cfa_loadings.png"), dpi=120)
plt.close(fig)
print("saved fig_cfa_loadings.png")

saved fig_cfa_loadings.png


![CFA Two-Factor Loadings](fig_cfa_loadings.png)

## IRT: Recovering Item Difficulty

CFA treats responses as continuous variables, but responses are actually binary—either right or wrong. Item Response Theory starts from this fact, assigning each item two parameters: discrimination a (how sensitive the item is to ability differences, corresponding to the slope of the curve) and difficulty b (the ability level needed for a 50% probability of answering correctly, corresponding to the horizontal shift of the curve); and each person an ability θ. Together these form a logistic probability curve `P = 1 / (1 + exp(−a·(θ − b)))`.

The falsification point here is clear: our data were generated using known a and b (the `truth` table from the start), so the IRT-estimated difficulty b **should recover the true ordering**. When `girth` is available, `sv.tl.irt` uses marginal maximum likelihood (MML); when not, it uses scipy for joint maximum likelihood—alternately updating examinee ability and item-wise (a, b). We place estimates and truth values side by side and quantify recovery using Spearman/Pearson correlations.

In [9]:
st_irt = sv.StudyState()
st_irt.write("sources", "datasets", resp)   # IRT 用原始 0/1 作答,不转连续
sv.tl.irt(st_irt)

irt_model = st_irt.models["irt"]
est_a = np.array(irt_model["a"])
est_b = np.array(irt_model["b"])
theta = np.array(irt_model["theta"])
true_a = truth["a"].to_numpy()
true_b = truth["b"].to_numpy()
print("backend:", irt_model["backend"],
      f"| {irt_model['n_persons']} 人 × {irt_model['n_items']} 题")

backend: scipy_jml_2pl | 400 人 × 10 题


Display the estimated (a, b) and true values side by side for each item. Focus on difficulty b: estimates should move in the same direction as true values and be similar in magnitude. Discrimination a also roughly matches, but note that one item (item5) has a pushed to the estimation ceiling of 6.0—this is typical boundary behavior for joint maximum likelihood when discrimination is very high and responses are nearly perfectly separable; MML backends typically do not exhibit such extremes. It is worth noting such boundary cases, but they do not affect recovery of the difficulty ordering.

In [10]:
pd.DataFrame({
    "item": irt_model["items"],
    "a_est": np.round(est_a, 3),
    "a_true": np.round(true_a, 3),
    "b_est": np.round(est_b, 3),
    "b_true": np.round(true_b, 3),
})

,item,a_est,a_true,b_est,b_true
0,item1,1.299,0.925,-1.645,-2.000
1,item2,1.872,1.462,-1.220,-1.556
2,item3,2.057,1.946,-1.196,-1.111
3,item4,1.898,1.631,-0.751,-0.667
4,item5,6.000,1.975,-0.188,-0.222
5,item6,1.843,1.447,0.268,0.222
6,item7,2.233,1.978,0.621,0.667
7,item8,2.082,1.788,1.014,1.111
8,item9,1.673,1.712,1.551,1.556
9,item10,1.548,1.855,2.393,2.000


Quantify recovery using correlation coefficients. The ordering of difficulty b is what we care most about—Spearman measures whether the orderings agree, Pearson measures whether magnitudes match.

In [11]:
sp_b = spearmanr(est_b, true_b).correlation
pe_b = pearsonr(est_b, true_b)[0]
sp_a = spearmanr(est_a, true_a).correlation
print(f"难度 b:Spearman(排序) = {sp_b:.3f} | Pearson(数量级) = {pe_b:.3f}")
print(f"区分度 a:Spearman(排序) = {sp_a:.3f}")
print("→ 难度排序被完美复原(Spearman = 1.000),数量级也高度一致(Pearson ≈ 0.99)。")

难度 b:Spearman(排序) = 1.000 | Pearson(数量级) = 0.989
区分度 a:Spearman(排序) = 0.709
→ 难度排序被完美复原(Spearman = 1.000),数量级也高度一致(Pearson ≈ 0.99)。


Two plots show the recovery. The left scatter plot displays **estimated vs. true difficulty**; points closer to the diagonal indicate better recovery. The right plot shows **item information curves** for three representative items (easiest / middle / hardest)—each item provides maximum information near the ability level matching its difficulty b, with peak value equal to a²/4, vividly illustrating how "hard items discriminate only for high-ability respondents".

In [12]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# 左:估计 b vs 真实 b
lo = min(est_b.min(), true_b.min()) - 0.3
hi = max(est_b.max(), true_b.max()) + 0.3
ax1.plot([lo, hi], [lo, hi], ls="--", color="gray", lw=1, label="y = x(完美复原)")
ax1.scatter(true_b, est_b, s=60, color="#3B6FB6", zorder=3)
for j, it in enumerate(irt_model["items"]):
    ax1.annotate(it, (true_b[j], est_b[j]), fontsize=7,
                 xytext=(3, 3), textcoords="offset points")
ax1.set_xlabel("真实难度 b (truth)")
ax1.set_ylabel("估计难度 b (2PL)")
ax1.set_title(f"难度复原  Pearson={pe_b:.3f}, Spearman={sp_b:.2f}")
ax1.legend(fontsize=9)

# 右:项目信息曲线(选难度分散的三道题)
theta_grid = np.linspace(-4, 4, 200)
pick = [0, len(est_a) // 2, len(est_a) - 1]  # 最易 / 居中 / 最难
for j in pick:
    p = 1.0 / (1.0 + np.exp(-est_a[j] * (theta_grid - est_b[j])))
    info = est_a[j] ** 2 * p * (1 - p)
    ax2.plot(theta_grid, info,
             label=f"{irt_model['items'][j]} (a={est_a[j]:.2f}, b={est_b[j]:+.2f})")
ax2.set_xlabel("能力 θ")
ax2.set_ylabel("Fisher 信息量")
ax2.set_title("项目信息曲线:每题在 θ≈b 处信息最大")
ax2.legend(fontsize=8)

fig.tight_layout()
fig.savefig(figpath("fig_irt.png"), dpi=120)
plt.close(fig)
print("saved fig_irt.png")

saved fig_irt.png


![IRT Difficulty Recovery and Information Curves](fig_irt.png)

As a side check, verify the distribution of examinee ability θ. The 2PL anchors θ to N(0,1) scale, so it should approximate the standard normal and be highly positively correlated with "total correct"—more correct answers implies higher ability, the most basic sanity check.

In [13]:
total_correct = resp.sum(axis=1).to_numpy()
r_theta_score = pearsonr(theta, total_correct)[0]
print(f"能力 θ:均值 = {theta.mean():+.3f}, 标准差 = {theta.std():.3f}")
print(f"θ 与总答对数的相关 = {r_theta_score:.3f}(接近 1,能力与总分同向)")

能力 θ:均值 = -0.015, 标准差 = 0.775
θ 与总答对数的相关 = 0.974(接近 1,能力与总分同向)


## SEM: Estimating Structural Paths and Recovering True Slopes

CFA and IRT both answer "do these items measure the same construct". SEM shifts the question: it cares about **paths between variables**. The classic special case is path analysis—all observed variables, no latent variables—treating a set of regression equations as a simultaneous system to estimate, yielding standardized path coefficients, R² for each equation, and global fit indices.

We estimate a structural path `y ~ x` on the multilevel data. Because the data were generated from `y = 1 + u + 2·x + ε`, **the true slope is β=2.0**, the falsification point for this step: SEM should recover it. When `semopy` is available, `sv.tl.sem` runs full ML-SEM; when not, it estimates equation-by-equation via OLS, honestly labeling the estimator as `path_analysis_ols`.

In [14]:
st_sem = sv.StudyState()
st_sem.write("sources", "datasets", multilevel)
sv.tl.sem(st_sem, paths={"y": ["x"]})   # 结构路径:y 由 x 预测

sem_model = st_sem.models["sem"]
sem_fit = st_sem.diagnostics["fit_indices"]
print("estimator:", sem_model["estimator"], "|", sem_model["backend"])
print("note     :", sem_model["note"])

estimator: path_analysis_ols | statsmodels/numpy OLS
note     : observed-variable path analysis (OLS per equation; no latent variables)


Examine the estimated path coefficient and equation R². The coefficient from `y ~ x` is the slope we aim to recover; compare it to the true value 2.0.

In [15]:
beta_hat = sem_model["coefficients"]["y"]["x"]
intercept = sem_model["coefficients"]["y"]["(intercept)"]
r2 = sem_model["r2"]["y"]
print(f"路径 y ~ x       = {beta_hat:+.4f}   ← 真值 β = 2.0")
print(f"截距 (intercept) = {intercept:+.4f}")
print(f"方程 y 的 R²      = {r2:.3f}")
print(f"\n→ 复原斜率 {beta_hat:.4f} vs 真值 2.0,误差仅 {abs(beta_hat - 2.0):.4f}。")

路径 y ~ x       = +2.0114   ← 真值 β = 2.0
截距 (intercept) = +1.2081
方程 y 的 R²      = 0.660

→ 复原斜率 2.0114 vs 真值 2.0,误差仅 0.0114。


Plot this path as a scatter plot with fitted line: gray points are (x, y) observations, red line is the SEM-estimated structural path `y = intercept + β·x`, blue dashed line is the data-generating process `y = 1 + 2·x`. The two lines nearly coincide, showing that the path coefficient is accurately recovered.

In [16]:
fig, ax = plt.subplots(figsize=(7, 5))
x = multilevel["x"].to_numpy()
y = multilevel["y"].to_numpy()

ax.scatter(x, y, s=14, alpha=0.35, color="#555555", label="观测 (x, y)")
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, intercept + beta_hat * xs, color="#C1544A", lw=2.2,
        label=f"SEM 路径:y = {intercept:.2f} + {beta_hat:.2f}·x")
ax.plot(xs, 1.0 + 2.0 * xs, color="#3B6FB6", lw=1.6, ls="--",
        label="真实结构:y = 1.00 + 2.00·x")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"SEM 路径分析:斜率复原  (R²={r2:.2f})")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(figpath("fig_sem_path.png"), dpi=120)
plt.close(fig)
print("saved fig_sem_path.png")

saved fig_sem_path.png


![SEM Path Analysis](fig_sem_path.png)

## Reproducible Evidence Chain

Each of the three models lives in its own `StudyState` (measurement model, IRT, and structural model, one each). Every state carries a provenance ledger—which function was called, which slots were read, where results were written—automatically recorded by socialverse after each call, no manual logging. `summary()` prints this ledger, making "where this conclusion came from, which data, which step" immediately clear. In social science, this chain of evidence is often as important as the conclusion itself.

In [17]:
print("=== CFA state ===")
print(st_cfa.summary())
print("\n=== IRT state ===")
print(st_irt.summary())
print("\n=== SEM state ===")
print(st_sem.summary())

=== CFA state ===
StudyState
  sources: datasets
  models: cfa
  diagnostics: fit_indices
  provenance: 1 step(s)

=== IRT state ===
StudyState
  sources: datasets
  models: irt
  diagnostics: item_info
  provenance: 1 step(s)

=== SEM state ===
StudyState
  sources: datasets
  models: sem
  diagnostics: fit_indices
  provenance: 1 step(s)


## Summary

We have walked through "inferring invisible constructs" from three complementary angles: CFA validates the pre-specified two-factor structure (three-metric suite CFI≈0.95 / RMSEA≈0.04 / SRMR≈0.06 all meet standards), IRT recovers item difficulty (ordering Spearman=1.000, magnitude Pearson≈0.99), SEM recovers the true slope of structural paths (β=2.0 recovered as 2.01). This chain benchmarks against R's `lavaan` (CFA/SEM), `mirt` (IRT), and Python's `semopy`—the underlying algorithms are their ML factor analysis, 2PL maximum likelihood, and path analysis.

Compared to stacking these packages in a script, socialverse adds two things: first, every dataset embeds ground truth, each step validates by comparing estimates to true values, verifying "did it recover what is known to be true" rather than "does it look like a good fit"—this incidentally exposes details like how IRT's joint maximum likelihood can push a to a boundary on very high-discrimination items, worth noting; second, a provenance ledger that runs throughout and is automatically recorded, making the entire analysis traceable and reproducible. The next tutorial [13_multilevel_survival](13_multilevel_survival.ipynb) turns to multilevel linear models and survival analysis.